# Section 4 - Baseline & logistic regression

**Goal of this notebook.** Build the first real models and, just as important, the
floor they have to beat. The plan:

1. **Naive baselines** - majority-class, and a simple "direct award = single bid"
   rule. Any real model must beat these.
2. **Feature preparation** - log-transform `value_eur`, one-hot encode the
   categoricals, stratified train/test split.
3. **Logistic regression**, fitted twice: on all contracts, and **with direct
   awards excluded** - the circularity experiment that Sections 1 and 3 pointed to.
4. **The math** - sigmoid, log-loss, and how to read the coefficients as effects
   on the odds of single-bidding, interpreted on the model's *own* output.

Metrics throughout: accuracy, precision, recall, F1, ROC-AUC - because for a
flag like this, accuracy alone misleads (Section 1). Findings and decisions are
collected in the closing cell.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

modelling = pd.read_csv("../data/processed/contracts-modelling.csv")

# single_bid is boolean in the cleaned table; cast to int for sklearn.
modelling["single_bid"] = modelling["single_bid"].astype(int)

DIRECT = "Пряко / без обявление"   # the near-circular direct-award procedure
target_rate = modelling["single_bid"].mean()
print(f"rows: {len(modelling):,} | single-bid rate: {target_rate:.1%}")

rows: 174,119 | single-bid rate: 44.9%


In [2]:
# A look at the modelling data.
modelling.head()

,id,unp,subject,authority,authority_eik,contractor,contractor_eik,kind,sector_code,procedure,signed_at,value_eur,eu_funded,bids_received,single_bid,type_group,region
0,e:00001-2021-0001:106742:_:eik:201472746:1,00001-2021-0001,Доставка и монтаж на каскадна инсталация от 6 ...,ИКОНОМИЧЕСКИ УНИВЕРСИТЕТ - ВАРНА,000083619,МЕГА ЕЛЕКТРОНИКС-АП ООД,201472746.0,company,42.0,Събиране на оферти,2021-07-16,32138.785068,0,3,0,образование,Варна
1,e:00001-2021-0003:9643:_:eik:131468980:1,00001-2021-0003,Предоставяне на далекосъобщителна услуга чрез ...,ИКОНОМИЧЕСКИ УНИВЕРСИТЕТ - ВАРНА,000083619,"""А1 БЪЛГАРИЯ"" ЕАД",131468980.0,company,64.0,Събиране на оферти,2021-08-26,35789.920392,0,1,1,образование,Варна
2,e:00001-2021-0006:15251:_:eik:103765686:1,00001-2021-0006,"Приготвяне, доставка на храна и осъществяване ...",ИКОНОМИЧЕСКИ УНИВЕРСИТЕТ - ВАРНА,000083619,ЗАЛИВА 47-СП АД,103765686.0,company,55.0,Пряко / без обявление,2021-10-11,409033.504957,0,3,0,образование,Варна
3,e:00001-2021-0008:25147:_:eik:831079085:1,00001-2021-0008,„Снабдяване с природен газ за отопление на обе...,ИКОНОМИЧЕСКИ УНИВЕРСИТЕТ - ВАРНА,000083619,"""Примагаз"" АД",831079085.0,company,9.0,Пряко / без обявление,2021-12-15,76693.782179,0,1,1,образование,Варна
4,e:00001-2021-0009:29019:_:eik:130533432:1,00001-2021-0009,„Снабдяване с природен газ за отопление на обе...,ИКОНОМИЧЕСКИ УНИВЕРСИТЕТ - ВАРНА,000083619,"""ОВЕРГАЗ МРЕЖИ"" АД",130533432.0,company,9.0,Пряко / без обявление,2022-01-19,255645.940598,0,1,1,образование,Варна


## 1. Naive baselines - the floor to beat

Two baselines. The **majority-class** predictor always guesses the most common
class (here "not single-bid", ~55%), so its accuracy is just the majority share -
and its recall for single-bid is zero. The **direct-award rule** predicts
single-bid whenever the procedure is a direct award; it is a deliberately dumb
rule that exploits the circularity from Section 1, and looking at *how* it scores
makes the circularity concrete.

In [3]:
# Baseline 1: majority class - always predict the most common class ("not single-bid").
# It never predicts single-bid, so it catches none of them: recall = 0.
majority_acc = 1 - target_rate
print(f"Baseline 1 - majority class:")
print(f"  always predicts 'not single-bid' → accuracy {majority_acc:.3f}, recall 0 (flags nothing)\n")

# Baseline 2: the "dumb rule" - predict single-bid iff the procedure is a direct award.
rule_pred = (modelling["procedure"] == DIRECT).astype(int)
y_all = modelling["single_bid"]

# Confusion matrix: the four cells every metric below is built from.
tn, fp, fn, tp = confusion_matrix(y_all, rule_pred).ravel()
print("Baseline 2 - direct-award rule:")
print(f"  flags {rule_pred.sum():,} contracts as single-bid (all the direct awards)")
print(f"  of {y_all.sum():,} contracts that truly are single-bid\n")
print(f"  true positives  (flagged & truly single-bid): {tp:,}")
print(f"  false positives (flagged but not single-bid): {fp:,}")
print(f"  false negatives (single-bid the rule missed): {fn:,}")
print(f"  true negatives  (correctly left unflagged):   {tn:,}\n")

# Each metric, shown as the fraction it actually is.
print(f"  precision = tp/(tp+fp) = {tp:,}/{tp+fp:,} = {tp/(tp+fp):.3f}  (of flagged, share truly single-bid)")
print(f"  recall    = tp/(tp+fn) = {tp:,}/{tp+fn:,} = {tp/(tp+fn):.3f}  (of all single-bid, share caught)")
print(f"  accuracy  = (tp+tn)/N  = {tp+tn:,}/{len(y_all):,} = {(tp+tn)/len(y_all):.3f}")

Baseline 1 - majority class:
  always predicts 'not single-bid' → accuracy 0.551, recall 0 (flags nothing)

Baseline 2 - direct-award rule:
  flags 20,953 contracts as single-bid (all the direct awards)
  of 78,158 contracts that truly are single-bid

  true positives  (flagged & truly single-bid): 16,854
  false positives (flagged but not single-bid): 4,099
  false negatives (single-bid the rule missed): 61,304
  true negatives  (correctly left unflagged):   91,862

  precision = tp/(tp+fp) = 16,854/20,953 = 0.804  (of flagged, share truly single-bid)
  recall    = tp/(tp+fn) = 16,854/78,158 = 0.216  (of all single-bid, share caught)
  accuracy  = (tp+tn)/N  = 108,716/174,119 = 0.624


The direct-award rule lands at **80% precision but only ~22% recall**: when
it fires it is usually right (direct awards really are ~80% single-bid) but it
stays silent on the vast majority of single-bid contracts that are *not* direct
awards. That split is the circularity in miniature - high precision bought purely
from the definitional overlap, no genuine prediction. A useful model has to do
much better on recall and overall discrimination than this.

## 2. Feature preparation

Numeric: `value_eur` is log-transformed (`log1p`, which handles the 146 exact
zeros), and `eu_funded` is already 0/1. Categorical: `kind`, `procedure`,
`type_group`, `region`, `sector_code` are one-hot encoded. **Excluded by design**
(leakage / circularity discipline from earlier sections): `bids_received` and
`single_bid` themselves, identifiers, and the contractor (known only post-award).
`procedure` is *kept* here on purpose - the whole point is to measure its effect,
not hide it.

In [4]:
def build_features(frame):
    """Return (X, y) for a given slice of the modelling table."""
    X_num = pd.DataFrame({
        "log_value": np.log1p(frame["value_eur"]),
        "eu_funded": frame["eu_funded"].values,
    })
    cats = frame[["kind", "procedure", "type_group", "region", "sector_code"]].astype(str)
    X_cat = pd.get_dummies(cats, drop_first=True, dtype=int)
    X = pd.concat([X_num.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)
    return X, frame["single_bid"].values

X_all, y = build_features(modelling)
print(f"feature matrix: {X_all.shape[0]:,} rows x {X_all.shape[1]} features")

# The two numeric features, plus a few of the one-hot columns, for 5 contracts.
sample_cols = ["log_value", "eu_funded"] + [c for c in X_all.columns if c.startswith("procedure_")][:3]

print("\nsample of features (5 rows, selected columns):")
print(X_all[sample_cols].head().to_string())

print("\ncolumn groups produced by one-hot encoding:")
for prefix in ["kind", "procedure", "type_group", "region", "sector_code"]:
    n = sum(c.startswith(prefix + "_") for c in X_all.columns)
    print(f"  {prefix:12s} → {n} columns")

feature matrix: 174,119 rows x 87 features

sample of features (5 rows, selected columns):
   log_value  eu_funded  procedure_Друго  procedure_Неизвестна  procedure_Открита
0  10.377850          0                0                     0                  0
1  10.485450          0                0                     0                  0
2  12.921555          0                0                     0                  0
3  11.247589          0                0                     0                  0
4  12.451553          0                0                     0                  0

column groups produced by one-hot encoding:
  kind         → 1 columns
  procedure    → 6 columns
  type_group   → 6 columns
  region       → 28 columns
  sector_code  → 44 columns


## 3. Logistic regression - with and without direct awards

The model is fitted twice. **With direct awards** uses all contracts. **Without
direct awards** drops the near-circular procedure entirely, so the model has to
predict single-bidding from genuine, non-definitional signal. Comparing the two
shows how much of the apparent performance came from the circular feature - the
experiment Sections 1 and 3 kept pointing to. Both use a stratified 70/30 split
(preserving the class balance) and standardised features.

In [5]:
def fit_logistic(frame, label):
    X, y = build_features(frame)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=42)
    scaler = StandardScaler(with_mean=False)  # with_mean=False: safe for sparse one-hot
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_tr_s, y_tr)
    pred = model.predict(X_te_s)
    prob = model.predict_proba(X_te_s)[:, 1]

    scores = {
        "accuracy":  accuracy_score(y_te, pred),
        "precision": precision_score(y_te, pred),
        "recall":    recall_score(y_te, pred),
        "f1":        f1_score(y_te, pred),
        "roc_auc":   roc_auc_score(y_te, prob),
    }
    print(f"--- {label}  (n={len(frame):,}, features={X.shape[1]}) ---")
    for k, v in scores.items():
        print(f"  {k:10s} {v:.3f}")
    return model, X.columns, scores, (y_te, pred)

In [6]:
m_with, cols_with, sc_with, (yte_w, pred_w) = fit_logistic(modelling, "WITH direct awards")
print()
m_wo, cols_wo, sc_wo, _ = fit_logistic(modelling[modelling["procedure"] != DIRECT],
                                       "WITHOUT direct awards")

--- WITH direct awards  (n=174,119, features=87) ---
  accuracy   0.663
  precision  0.688
  recall     0.455
  f1         0.548
  roc_auc    0.712

--- WITHOUT direct awards  (n=153,166, features=86) ---
  accuracy   0.638
  precision  0.594
  recall     0.305
  f1         0.403
  roc_auc    0.664


In [7]:
# Side-by-side comparison vs the majority baseline.
comparison = pd.DataFrame({
    "majority_baseline": {"accuracy": majority_acc, "precision": 0, "recall": 0,
                          "f1": 0, "roc_auc": 0.5},
    "logistic_with_direct": sc_with,
    "logistic_without_direct": sc_wo,
}).T
print(comparison.round(3).to_string())

                         accuracy  precision  recall     f1  roc_auc
majority_baseline           0.551      0.000   0.000  0.000    0.500
logistic_with_direct        0.663      0.688   0.455  0.548    0.712
logistic_without_direct     0.638      0.594   0.305  0.403    0.664


**The headline result.** With direct awards, logistic regression reaches
**ROC-AUC 0.71** and F1 **0.55**, clearly beating the majority baseline. Drop the
direct-award procedure and performance falls - **ROC-AUC 0.66**, F1 **0.40** - but
it does **not** collapse to chance (0.5). 

Both halves of that matter. The drop confirms part of the model's strength came
from the circular direct-award signal - honest to report, and exactly why the
experiment is run. But the floor at 0.66 confirms there is *real* predictive
signal beyond direct awards: the model is not merely a direct-award detector,
consistent with Section 1's finding that ~78% of single-bid contracts are not
direct awards. The genuine, harder problem - predicting single-bidding among
competitive procedures - is learnable, just less sharply.

In [8]:
# Confusion matrix for the with-direct model (test set).
cm = confusion_matrix(yte_w, pred_w)
print("confusion matrix (rows = actual, cols = predicted):")
print(pd.DataFrame(cm, index=["actual_not", "actual_single"],
                   columns=["pred_not", "pred_single"]).to_string())

confusion matrix (rows = actual, cols = predicted):
               pred_not  pred_single
actual_not        23958         4830
actual_single     12780        10668


## 4. The math, read on this model

**The model.** Logistic regression predicts the probability of single-bidding by
passing a linear combination of the features through the sigmoid:

$$ P(\text{single bid}=1 \mid x) = \sigma(z) = \frac{1}{1 + e^{-z}}, \qquad
   z = w_0 + w_1 x_1 + \dots + w_m x_m $$

The sigmoid squashes any real $z$ into $(0, 1)$, so the output reads as a
probability. A contract is then flagged single-bid if that probability passes a
threshold (0.5 by default).

**Training - log-loss.** The weights $w$ are chosen to minimise the
log-loss (binary cross-entropy) over the training contracts:

$$ J(w) = -\frac{1}{n} \sum_{i=1}^{n}
   \Big[ y_i \ln \hat{y}_i + (1 - y_i)\ln(1 - \hat{y}_i) \Big] $$

where $\hat{y}_i = \sigma(z_i)$. This penalises confident wrong predictions
heavily, which is what pushes the weights toward values that separate single-bid
from competitive contracts.

**Reading the coefficients.** Each weight is an effect on the **log-odds** of
single-bidding; exponentiating gives an **odds ratio**. A positive $w_j$ means
feature $j$ raises the odds of a single bid; $e^{w_j}$ is the multiplicative
factor. The cell below pulls the largest-magnitude coefficients from the
*with-direct* model and reads them this way.

In [10]:
# Coefficients of the with-direct model, as odds ratios.
scaler_full = StandardScaler(with_mean=False).fit(
    train_test_split(X_all, y, test_size=0.30, stratify=y, random_state=42)[0])
coef = pd.Series(m_with.coef_[0], index=cols_with)
top = pd.concat([coef.sort_values().tail(6), coef.sort_values().head(4)])
table = pd.DataFrame({"coef_log_odds": top, "odds_ratio": np.exp(top)})
table = table.sort_values("odds_ratio", ascending=False)

# Map sector_code dummies to official CPV names for readability (display only;
# the trained model and cols_with are untouched). Labels come from notebook 00.
ref = pd.read_csv("../data/reference/cpv_sectors.csv")
labels_en = dict(zip(ref.sector_code, ref.label_en))

codes_in_table = {int(float(f.removeprefix("sector_code_")))
                  for f in table.index if f.startswith("sector_code_")}
missing = sorted(codes_in_table - set(labels_en))
assert not missing, f"Sector codes with no official CPV label: {missing}"

def pretty(feat):
    if feat.startswith("sector_code_"):
        code = int(float(feat.removeprefix("sector_code_")))
        return f"CPV {code}: {labels_en[code]}"
    return feat  # procedure_ / type_group_ rows left as-is

table.index = table.index.map(pretty)
print(table.round(3).to_string())

                                                                                           coef_log_odds  odds_ratio
procedure_Пряко / без обявление                                                                    0.455       1.577
type_group_образование                                                                             0.219       1.244
type_group_община                                                                                  0.198       1.219
CPV 60: Transport services (excl. Waste transport)                                                 0.136       1.146
type_group_друго                                                                                   0.115       1.122
CPV 72: IT services: consulting, software development, Internet and support                        0.108       1.114
procedure_Състезание                                                                              -0.256       0.774
CPV 45: Construction work                                       

As expected, the **direct-award procedure carries the largest positive
coefficient by far** - it multiplies the odds of single-bidding more than any
other feature, which is the coefficient-level signature of the circularity. After
it, the genuine signals appear: education and municipality authority types push
the odds up, while open/competitive procedures and certain sectors (construction
work, business services) push them down. This is the feature story the project has
been building toward - the model leans hardest on the circular feature, which is
precisely why removing it lowers the scores, and precisely why both numbers are
reported.

## Section 4 - findings & decisions

**Baselines (the floor):**
- Majority-class accuracy **0.55**, recall 0 - useless as a flag, but the
  accuracy number any real model must clear.
- Direct-award rule: **80% precision, 22% recall** - high precision bought purely
  from the definitional overlap, near-zero genuine coverage. The circularity, made
  concrete.

**Logistic regression:**
- **With direct awards:** ROC-AUC **0.71**, F1 **0.55**, accuracy **0.66** -
  beats the majority baseline, a real first model.
- **Without direct awards:** ROC-AUC **0.66**, F1 **0.40** - lower, but well above
  chance. Part of the performance was the circular signal (honest to report);
  the rest is genuine, so the model is *not* just a direct-award detector.

**The math** is laid out on the model's own output: sigmoid for the probability,
log-loss as the training objective, coefficients read as odds ratios - with the
direct-award procedure showing the largest odds ratio, the coefficient-level form
of the circularity.

**Decisions carried forward:**
- Report **both** with- and without-direct-award results everywhere; the gap is a
  finding, not something to hide.
- `eu_funded`, `value_eur` (log), `type_group`, `region`, `sector_code` all
  contribute genuine signal; keep them for the tree models.
- Logistic regression is the linear floor; the question for Section 5 is whether
  trees/ensembles capture interactions it misses.

**Next (Section 5):** decision tree and random forest /
boosting, same metrics, same with/without discipline, compared against this
logistic baseline in one table.